In [1]:
# 1. Clone the repository
!git clone https://github.com/Anshita00000000/elec-cons-forecasting.git

Cloning into 'elec-cons-forecasting'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 65 (delta 10), reused 47 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 12.33 MiB | 32.71 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [4]:
import os
path = "/content/elec-cons-forecasting"

if os.path.exists(path):
    %cd {path}
    print("Successfully moved to directory:", os.getcwd())
else:
    print("Folder not found. Please check the sidebar to see the exact folder name.")

/content/elec-cons-forecasting
Successfully moved to directory: /content/elec-cons-forecasting


In [ ]:
# Now try installing requirements again
!pip install -r requirements.txt

In [6]:
import os
from getpass import getpass
os.environ['EIA_API_KEY'] = getpass('Enter your EIA API Key: ')

# Run the pipeline
!python data_pipeline.py

Enter your EIA API Key: ··········
Fetching EIA electricity sales data...
  EIA pivot shape: (301, 5)
Fetching NOAA temp_max_f data...
  NOAA temp_max_f: 301 rows
Fetching NOAA temp_min_f data...
  NOAA temp_min_f: 301 rows
  Temperature df shape: (301, 6)
Merging electricity and temperature datasets...
  Merged df shape: (299, 12)
Validating master dataframe...
  No missing values — OK
Traceback (most recent call last):
  File "/content/elec-cons-forecasting/data_pipeline.py", line 379, in <module>
    validate_and_save(master, "data/master_df.csv")
  File "/content/elec-cons-forecasting/data_pipeline.py", line 320, in validate_and_save
    assert df["month"].min() == "2001-01", f"Unexpected start month: {df['month'].min()}"
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: Unexpected start month: 2001-03


In [7]:
print("--- Running Feature Engineering ---")
!python feature_engineering.py

--- Running Feature Engineering ---
Loaded master_df: (303, 12)
After dropping first 24 rows: 279 rows remaining — OK
  features_with_temp: (279, 44), no missing values — OK
  features_lag_only: (279, 41), no missing values — OK

Train/test split (target: sales_total):
  features_with_temp — train: 252 rows (2003-01 → 2023-12)
  features_with_temp — test:  27 rows (2024-01 → 2026-03)
  features_lag_only  — train: 252 rows (2003-01 → 2023-12)
  features_lag_only  — test:  27 rows (2024-01 → 2026-03)

  Saved → data/features_with_temp.csv
  Saved → data/features_lag_only.csv

Feature summary:
  Total lag/rolling features per target: 9 (lag 1,2,3,6,12,24 + roll3,roll12,roll3_std)
  Sectors covered: sales_total, sales_COM, sales_IND, sales_RES
  Calendar features: month_num, year, quarter
  Temperature features (with_temp only): temp_avg_f, HDD, CDD
  features_with_temp columns: 44 (incl. month + target)
  features_lag_only columns:  41 (incl. month + target)

Part 3 complete. feature_engi

In [8]:
print("--- Training Models (This will take a while) ---")
!python models.py

--- Training Models (This will take a while) ---
E0000 00:00:1776422562.557176    3027 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776422562.564436    3027 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776422562.583248    3027 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776422562.583280    3027 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776422562.583284    3027 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776422562.583288    3027 computation_placer.cc:177] computat

In [9]:
import urllib
# This gets your public IP fory the localtunnel bypass
print("External IP (Copy this for the Tunnel website):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

# Run streamlit
!nohup streamlit run app.py & npx localtunnel --port 8501

External IP (Copy this for the Tunnel website): 34.168.229.37
nohup: appending output to 'nohup.out'
⠙⠹⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://breezy-days-strive.loca.lt
^C


In [ ]:
# 1. Install Cloudflared
#!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
#!dpkg -i cloudflared-linux-amd64.deb

# 2. Kill old processes
#!pkill streamlit
#!pkill cloudflared

# 3. Start Streamlit
#!nohup streamlit run app.py --server.port 8501 &

# 4. Start Cloudflared tunnel
#!cloudflared tunnel --url http://localhost:8501